# 03 — Modelos base

Este notebook documentará M1–M3 paso a paso. Por ahora completa **M1: preparación de experimentos**.

Un **experimento** es una ejecución con datos, variables y parámetros definidos. Su registro permite saber exactamente cómo se obtuvo un resultado.

## 1. Configuración única

La semilla y las rutas se guardan en `configs/modeling.yaml`. Una **semilla** es un número que permite repetir los mismos pasos aleatorios.

In [ ]:
from src.evaluation.experiment import load_experiment_config, set_seed

config = load_experiment_config()
config

## 2. Comprobar la semilla

Al reiniciar la misma semilla, Python y NumPy deben producir los mismos números. Esto no garantiza que todo modelo sea idéntico, pero elimina una fuente común de variación.

In [ ]:
import random
import numpy as np

seed = config['experiment']['seed']
set_seed(seed)
primera_prueba = (random.random(), np.random.random())

set_seed(seed)
segunda_prueba = (random.random(), np.random.random())

assert primera_prueba == segunda_prueba
primera_prueba

## 3. Información mínima de cada ejecución

Cada ejecución registrará en MLflow:

- versión del código en Git;
- versión de los datos en DVC;
- objetivo y periodo evaluado;
- vista completa o sin texto compartido;
- variables de entrada;
- semilla;
- ejecución local o en Khipu;
- parámetros y métricas del modelo.

Una **métrica** es un número usado para evaluar un resultado. Por ejemplo, Macro-F1 para T1.

In [ ]:
from src.evaluation.experiment import build_run_record

registro_ejemplo = build_run_record(
    config=config,
    run_name='ejemplo-no-publicar',
    stage='M1',
    target='experiment_setup',
    split='not_applicable',
    view='not_applicable',
    features=[],
)
registro_ejemplo

## 4. Ejecuciones en Khipu

Los nodos SLURM no tienen acceso a internet. El trabajo guarda primero un registro JSON local. Al terminar, ese registro se publica en MLflow desde el nodo de acceso.

Esto separa dos acciones:

1. **Ejecutar:** calcular el resultado dentro de SLURM.
2. **Publicar:** enviar parámetros, métricas y artefactos pequeños a MLflow.

Los artefactos grandes, como embeddings o índices FAISS, se guardarán con DVC.

## Siguiente paso

En M2 se fijarán los periodos y métricas antes de entrenar. En M3 este mismo notebook comparará referencias simples contra las cuales deberán demostrar valor los modelos posteriores.